# Module 38: Unit Testing Spark Code with Pytest & Test Data Builders

## 1. What It Is
Unit testing in Apache Spark involves validating that business transformation functions produce correct DataFrame outputs across edge cases, null values, and boundary conditions.
Using **Pytest** with a session-scoped `SparkSession` fixture allows running hundreds of unit tests fast without paying the multi-second JVM initialization cost on every test.

## 2. Why It Matters
Untested Spark pipelines fail silently in production: subtle null pointer exceptions, wrong join cardialities, or silent column drops often go unnoticed until dashboards show corrupted metrics.
Structuring Spark code into pure, testable functions (`def transform_fn(df) -> df`) enables comprehensive CI/CD test automation.

## 3. Concept Diagram
```
  Pytest Test Runner
          │
          ▼
  @pytest.fixture(scope="session") ──► Spins up JVM & SparkSession ONCE (Fast!)
          │
          ├──► test_1: Verifies transformation on mock builder data
          ├──► test_2: Asserts schema and column data types
          └──► test_3: Validates null handling with pytest.approx()
```

## 7. Common Mistakes & Gotchas
- **Spinning up a SparkSession per test function**: Creating a new session inside each test (`scope="function"`) adds 3-5 seconds of JVM startup overhead per test. Always use `scope="session"`.
- **Comparing DataFrames with `.collect() == .collect()`**: In distributed systems, partition ordering is non-deterministic. Calling `df1.collect() == df2.collect()` fails if row order differs. Always sort by primary key or use `assertDataFrameEqual`.
- **Reading external disk files in unit tests**: Unit tests should be hermetic and fast. Use in-memory mock data builders (`spark.createDataFrame(mock_data)`) rather than reading large files from S3 or disk.

## 8. Official Documentation
- PySpark Testing Utilities: https://spark.apache.org/docs/latest/api/python/reference/pyspark.testing.html

In [ ]:
import sys
import os
import pytest

# Project root setup (compatible with both standalone execution and Jupyter notebooks)
if "__file__" in locals():
    project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), "../.."))
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), "../..") if "notebooks" in os.getcwd() else os.getcwd())
    os.chdir(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)
from scripts.spark_session import get_spark_session
from pyspark.sql.functions import col, when, round as _round

# 1. Pure Transformation Business Function (Separated from I/O)
def calculate_loyalty_rewards(df_customers, df_orders):

Pure business logic function. Accepts input DataFrames, returns transformed DataFrame.
    Calculates bonus loyalty points based on customer spending tier.